# 06　VLAの学習と評価の総復習

05で集めた成功データを使い、04と同じLoRAの方法でSmolVLAを学習します。最後に02で使ったLIBEROで、元のモデルと学習後のモデルを同じ条件で動かし、成功数と動画を見比べます。

05でデータを変換し、最後の保存セルで表示される「06のDATASET_ROOT」を控えておきます。

1. Google Driveの「マイドライブ」に `vla_workshop` フォルダを作り、その中に `06_collected_finetuning` をフォルダごと置きます。
2. このノートブックをColabで開き、ランタイムを **T4 GPU** にします。別の場所に置いた場合は、最初のコードセルの `LESSON_DIR` を変更します。
3. 上から順にセルを実行します。

同じフォルダに置くファイル

- `common.py`
- `evaluation.json`
- `libero_env.py`
- `prepare.py`
- `requirements.txt`
- `robot_io.py`
- `smolvla.py`
- `train.json`
- `training.py`

In [ ]:
# @title この回の配布ファイルを準備する
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

from google.colab import drive, files

USE_DRIVE = True  # @param {type:"boolean"}
LESSON_DIR = "/content/drive/MyDrive/vla_workshop/06_collected_finetuning"  # @param {type:"string"}
REQUIRED_FILES = [
    "common.py",
    "evaluation.json",
    "libero_env.py",
    "prepare.py",
    "requirements.txt",
    "robot_io.py",
    "smolvla.py",
    "train.json",
    "training.py",
]

if USE_DRIVE:
    drive.mount("/content/drive")
    source = Path(LESSON_DIR)
else:
    # ipynb以外の今回の配布ファイルを、まとめて選んでアップロードします。
    os.chdir("/content")
    files.upload()
    source = Path("/content")
missing = [name for name in REQUIRED_FILES if not (source / name).is_file()]
if missing:
    raise FileNotFoundError(f"配布ファイルが不足: {missing}。LESSON_DIRを確認してください。")
PROJECT = Path("/content/lesson_06")
PROJECT.mkdir(exist_ok=True)
for name in REQUIRED_FILES:
    shutil.copy2(source / name, PROJECT / name)
os.chdir(PROJECT)
print("この回の作業場所:", PROJECT)

## ライブラリの準備

次のセルで、この回に必要なライブラリを準備します。初回は数分かかることがあります。

In [ ]:
# @title ライブラリを準備する
LESSON_ENV = {
    **os.environ,
    "MPLBACKEND": "Agg",
    "PYTHONPATH": str(PROJECT),
    "PYTHONNOUSERSITE": "1",
}
command = [sys.executable, str(PROJECT / "prepare.py")]

subprocess.run(command, env=LESSON_ENV, check=True)
PYTHON = PROJECT / ".venv/bin/python"


def run_python(program):
    """セルに書いたPythonを、この回の専用Pythonで実行する。"""
    last_result = None
    with subprocess.Popen(
        [str(PYTHON), "-u", "-c", program],
        cwd=PROJECT,
        env=LESSON_ENV,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    ) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
            if line.startswith("Results: "):
                last_result = Path(line.removeprefix("Results: ").strip())
        if process.wait():
            raise RuntimeError("実行に失敗しました。直前のエラーを確認してください。")
    return last_result

## モデルを読み込み、行動を予測する処理

ここでは後のセルで使う処理を用意します。モデルを読み込むのは、実習を実行するセルです。

In [ ]:
MODEL_CODE = r"""
import torch
from lerobot.policies.factory import make_pre_post_processors
from lerobot.policies.smolvla import processor_smolvla  # noqa: F401

from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy

from smolvla import checkpoint_inputs, observation_batch


def load_model(settings, image_size=256):
    checkpoint, config, vlm_repo = checkpoint_inputs(settings, image_size)
    policy = SmolVLAPolicy.from_pretrained(checkpoint, config=config, strict=True).eval()
    # 正規化・トークン化・行動の逆正規化はチェックポイント付属のものを使います。
    pre, post = make_pre_post_processors(
        config,
        pretrained_path=checkpoint,
        preprocessor_overrides={
            "device_processor": {"device": config.device},
            "tokenizer_processor": {"tokenizer_name": config.vlm_model_name},
        },
        postprocessor_overrides={"device_processor": {"device": "cpu"}},
    )
    return policy, pre, post, vlm_repo


def predict_action(policy, pre, post, obs, instruction, settings):
    batch = observation_batch(obs, instruction, settings)
    with torch.inference_mode():
        action = post(policy.select_action(pre(batch)))
    return action.squeeze(0).detach().cpu().numpy()

"""

## LIBEROでの評価処理

カメラ画像などの観測から行動を予測し、シミュレータを1ステップ進めます。評価を実行するセルでこの処理を使います。

In [ ]:
EVALUATION_CODE = r"""
import hashlib

import imageio.v2 as imageio
import numpy as np

from common import output_directory, write_json
from libero_env import ASSETS_REVISION, open_scene, scene_cases
from robot_io import validate_action


def evaluate(settings, split):
    policy, pre, post, _ = load_model(settings["policy"])
    folder = output_directory("libero_" + split)
    episodes = []

    for task, initial in scene_cases(settings, split):
        print(f"タスク{task}・開始時の配置{initial}を評価中", flush=True)
        seed = settings["seed"] + task * 10000 + initial
        torch.manual_seed(seed)
        np.random.seed(seed)
        policy.reset()
        pre.reset()
        post.reset()
        env, obs, instruction = open_scene(settings, task, initial, seed)
        try:
            state_hash = hashlib.sha256(np.asarray(env.get_sim_state()).tobytes()).hexdigest()
            video_path = folder / f"task_{task:02d}_init_{initial:03d}.mp4"
            success = False
            with imageio.get_writer(video_path, fps=20) as video:
                for step in range(settings["max_steps"]):
                    video.append_data(obs["agentview_image"][::-1, ::-1])
                    action = predict_action(policy, pre, post, obs, instruction, settings["policy"])
                    obs, _, done, _ = env.step(validate_action(action))
                    success = bool(env.check_success())
                    if success or done:
                        break
                video.append_data(obs["agentview_image"][::-1, ::-1])
            episodes.append({
                "task_id": task,
                "initial_state": initial,
                "seed": seed,
                "initial_state_sha256": state_hash,
                "instruction": instruction,
                "steps": step + 1,
                "success": success,
                "video": video_path.name,
            })
        finally:
            env.close()

    successes = sum(episode["success"] for episode in episodes)
    report = {
        "split": split, "settings": settings, "assets_revision": ASSETS_REVISION,
        "trials": len(episodes), "successes": successes, "success_rate": successes / len(episodes),
        "episodes": episodes,
    }
    write_json(folder / "report.json", report)
    print(f"Results: {folder}", flush=True)
    return folder

"""

## 05で保存したデータの場所を指定する

05の保存セルに表示された「06のDATASET_ROOT」のパスを、下の `DATASET_ROOT` に貼り付けます。変換した成功試行のうち、最後の1件は検証lossの確認に使い、残りで学習します。初期設定では、学習率は `0.0001`、LoRAのランクは `8` です。この回では10回更新します。

In [ ]:
# @title 05で保存したデータの場所を指定する
DATASET_ROOT = ""  # @param {type:"string"}

drive.mount("/content/drive")
info_path = Path(DATASET_ROOT) / "meta/info.json"
if not DATASET_ROOT or not info_path.is_file():
    raise FileNotFoundError("05で変換したdatasetフォルダを指定してください")
episode_count = json.loads(info_path.read_text())["total_episodes"]
if episode_count < 3:
    raise ValueError("06では05で集めた成功試行が3件以上必要です")

config = json.loads((PROJECT / "train.json").read_text())
config["dataset"]["root"] = DATASET_ROOT
config["dataset"]["train_episodes"] = list(range(episode_count - 1))
config["dataset"]["validation_episodes"] = [episode_count - 1]
(PROJECT / "active_train.json").write_text(json.dumps(config))
print("学習用:", config["dataset"]["train_episodes"])
print("検証用:", config["dataset"]["validation_episodes"])

## 集めたデータでLoRAを学習する

04と同じ方法で、元の重みを固定してLoRAを更新します。ここでは、05で選別した記録を教師データとして使います。

In [ ]:
result = run_python(
    MODEL_CODE
    + r"""
import json
from lerobot.utils.collate import lerobot_collate_fn
from peft import LoraConfig, get_peft_model

from common import output_directory, read_json, write_json
from training import load_training_data, save_model

config = read_json("active_train.json")
torch.manual_seed(config["seed"])
base_policy, pre, post, vlm_repo = load_model(config["policy"])
training, validation = load_training_data(config["dataset"], base_policy)

# Action ExpertのAttentionにLoRAを追加し、元の重みは固定します。
lora = config["lora"]
lora_config = LoraConfig(
    r=lora["rank"],
    lora_alpha=lora["alpha"],
    target_modules=r"model\.vlm_with_expert\.lm_expert\..*\.(q|v)_proj",
    bias="none",
    lora_dropout=0.0,
)
policy = get_peft_model(base_policy, lora_config)
trainable = [
    (name, parameter)
    for name, parameter in policy.named_parameters()
    if parameter.requires_grad
]
if not trainable or any("lora_" not in name for name, _ in trainable):
    raise RuntimeError("LoRAの行列だけが学習対象かを確認してください")
parameters = [parameter for _, parameter in trainable]
trainable_count = sum(parameter.numel() for parameter in parameters)
total_count = sum(parameter.numel() for parameter in policy.parameters())
print(f"学習するパラメータ: {trainable_count:,} / {total_count:,}", flush=True)

# 学習率lrを変えると、同じ勾配でも1回の更新の大きさが変わります。
optimizer = torch.optim.AdamW(parameters, lr=config["learning_rate"], weight_decay=0.0)
batch_size = config["batch_size"]
if min(config["steps"], batch_size) < 1:
    raise ValueError("stepsとbatch_sizeは1以上にしてください")

held_out = lerobot_collate_fn([validation[i % len(validation)] for i in range(batch_size)])
device = parameters[0].device
rng_devices = [device] if device.type == "cuda" else []


def measure_validation_loss():
    policy.eval()
    # 同じ記録と乱数で測り、学習前後のlossを比べます。
    with torch.random.fork_rng(devices=rng_devices), torch.inference_mode():
        torch.manual_seed(config["seed"] + 1)
        loss, _ = policy(pre(held_out))
    return loss.item() if torch.isfinite(loss) else None


validation_before = measure_validation_loss()
if validation_before is None:
    raise RuntimeError("学習前の検証lossが有限ではありません")

policy.train()
losses = []
status = "完了"
for step in range(config["steps"]):
    indices = torch.randint(len(training), (batch_size,)).tolist()
    batch = lerobot_collate_fn([training[index] for index in indices])
    optimizer.zero_grad()
    loss, _ = policy(pre(batch))
    if not torch.isfinite(loss):
        status = "lossが有限でないため停止"
        break
    loss.backward()
    grad_norm = torch.nn.utils.clip_grad_norm_(parameters, config["max_grad_norm"])
    if not torch.isfinite(grad_norm):
        status = "勾配が有限でないため停止"
        break
    optimizer.step()
    row = {"step": step + 1, "train_loss": loss.item(), "grad_norm": grad_norm.item()}
    losses.append(row)
    print(row, flush=True)

validation_after = measure_validation_loss() if status == "完了" else None
if status == "完了" and validation_after is None:
    status = "学習後の検証lossが有限でないため停止"

folder = output_directory("finetuning")
(folder / "losses.jsonl").write_text("".join(json.dumps(row) + "\n" for row in losses))
checkpoint = None
if status == "完了":
    # 推論で読み直せるように、学習した補正を元の重みに足して保存します。
    merged_policy = policy.merge_and_unload(safe_merge=True)
    checkpoint = folder / "pretrained_model"
    save_model(merged_policy, pre, post, vlm_repo, checkpoint)

write_json(folder / "training.json", {
    "config": config,
    "status": status,
    "steps_completed": len(losses),
    "trainable_parameters": trainable_count,
    "total_parameters": total_count,
    "validation_before": validation_before,
    "validation_after": validation_after,
    "checkpoint": str(checkpoint) if checkpoint is not None else None,
})
print(status, flush=True)
print(f"Results: {folder}", flush=True)
"""
)

training_report = json.loads((result / "training.json").read_text())
print("検証loss・学習前:", training_report["validation_before"])
print("検証loss・学習後:", training_report["validation_after"])
print("学習したパラメータ数:", training_report["trainable_parameters"])

## 比較結果をまとめる処理

同じタスク・初期状態の試行を対応させ、成功率の差を計算します。

In [ ]:
COMPARISON_CODE = r"""
from os.path import relpath

from common import output_directory, read_json, write_json

settings = read_json("comparison_settings.json")
for key, value in settings["baseline"].items():
    if key != "policy" and settings["candidate"][key] != value:
        raise ValueError(f"評価条件が一致しません: {key}")

print("元のモデルを評価します。", flush=True)
baseline = evaluate(settings["baseline"], SPLIT)
print("学習後のモデルを評価します。", flush=True)
candidate = evaluate(settings["candidate"], SPLIT)
before = read_json(baseline / "report.json")
after = read_json(candidate / "report.json")
folder = output_directory("comparison_" + SPLIT)
pairs = []
for left, right in zip(before["episodes"], after["episodes"], strict=True):
    for key in ("task_id", "initial_state", "seed", "initial_state_sha256", "instruction"):
        if left[key] != right[key]:
            raise ValueError(f"試行が対応していません: {key}")
    pairs.append({
        "task_id": left["task_id"], "initial_state": left["initial_state"],
        "baseline": left["success"], "candidate": right["success"],
        "baseline_video": relpath(baseline / left["video"], folder),
        "candidate_video": relpath(candidate / right["video"], folder),
    })
write_json(folder / "comparison.json", {
    "split": SPLIT, "trials": len(pairs),
    "baseline_success_rate": before["success_rate"],
    "candidate_success_rate": after["success_rate"],
    "difference_percentage_points": 100 * (after["success_rate"] - before["success_rate"]),
    "pairs": pairs,
})
print(f"Results: {folder}", flush=True)
"""

## 学習前後のモデルを比べる

元のSmolVLAと学習後のSmolVLAを、05の収集には使っていない同じ3つの初期配置で動かします。タスクはどちらもタスク0です。ColabのT4で比較にかかった時間は約30分でした。

成功数と、初期配置ごとの成否を比べます。「元のモデルでは未達、学習後は成功」になった配置があれば、まずその動画を表示します。別の10配置で確認したい場合は、`FINAL_TEST` にチェックを入れます。

In [ ]:
# @title 元のモデルと学習したモデルを比べる
from copy import deepcopy

from IPython.display import Markdown, Video, display

FINAL_TEST = False  # @param {type:"boolean"}

checkpoint = training_report["checkpoint"]
if checkpoint is None:
    raise RuntimeError(f"保存したモデルがありません: {training_report['status']}")

baseline = json.loads((PROJECT / "evaluation.json").read_text())
baseline["policy"] = training_report["config"]["policy"]
candidate = deepcopy(baseline)
candidate["policy"].update(path=checkpoint, revision=None)
settings = {"baseline": baseline, "candidate": candidate}
(PROJECT / "comparison_settings.json").write_text(json.dumps(settings))

split = "test" if FINAL_TEST else "validation"
comparison_result = run_python(
    MODEL_CODE + EVALUATION_CODE + f"\nSPLIT = {split!r}\n" + COMPARISON_CODE
)
comparison = json.loads((comparison_result / "comparison.json").read_text())
pairs = comparison["pairs"]
before = sum(pair["baseline"] for pair in pairs)
after = sum(pair["candidate"] for pair in pairs)
print(f"元のモデル: {before} / {len(pairs)} 試行成功")
print(f"学習後のモデル: {after} / {len(pairs)} 試行成功")
print(f"成功率の差: {comparison['difference_percentage_points']:+.1f}ポイント")

def outcome(success):
    return "成功" if success else "未達"


rows = [
    "| 開始時の配置 | 元のモデル | 学習後のモデル |",
    "| ---: | --- | --- |",
]
for pair in pairs:
    rows.append(
        f"| {pair['initial_state']} | {outcome(pair['baseline'])} | "
        f"{outcome(pair['candidate'])} |"
    )
display(Markdown("\n".join(rows)))

improved = [pair for pair in pairs if not pair["baseline"] and pair["candidate"]]
print(f"未達から成功に変わった試行: {len(improved)} 件")
changed = improved[0] if improved else next(
    (pair for pair in pairs if pair["baseline"] != pair["candidate"]), pairs[0]
)
label = f"{outcome(changed['baseline'])} → {outcome(changed['candidate'])}"
display(Markdown(f"### 開始時の配置 {changed['initial_state']} の動画（{label}）"))
display(Markdown("元のモデル"))
display(Video(str(comparison_result / changed["baseline_video"]), embed=True))
display(Markdown("学習後のモデル"))
display(Video(str(comparison_result / changed["candidate_video"]), embed=True))

## まとめ

05で採用した試行のうち、何件を学習に使いましたか。元のモデルと学習後のモデルで成功数を比べ、同じ初期配置の動画を見て気づいたことを一つ書いてください。

## 結果をDriveに保存する

学習したモデル、loss、学習前後の比較結果と動画を保存します。

In [ ]:
# @title 結果を保存する
from datetime import UTC, datetime
from uuid import uuid4

RESULTS_DIR = "/content/drive/MyDrive/vla_workshop/results"  # @param {type:"string"}
drive.mount("/content/drive")
destination = Path(RESULTS_DIR)
destination.mkdir(parents=True, exist_ok=True)
name = (
    "06_collected_finetuning_"
    + datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ")
    + "_"
    + uuid4().hex[:8]
)
saved = destination / name
shutil.copytree(PROJECT / "outputs", saved)
print("保存先:", saved)
for model in sorted(saved.glob("*/pretrained_model/config.json")):
    print("保存したモデル:", model.parent)